# Bulk editor for ArcGIS Online Item Description pages


**Welcome!**  
This notebook helps you scan, review, and update ArcGIS Online items at scale. It focuses on the Terms of Use section, stored in the `licenseInfo` field, and looks for text or HTML that you may want to replace.
This version bundles `helper_functions.py` and `Esri_ToU.html` template directly into the notebook, though you can modify both input and output as you progress. A review webpage is produced that lets you see what will change before you make any edits, and you can selectively choose to edit items from the report.
*** BE CAUTIOUS WITH ANY TOOL LIKE THIS THAT BULK EDITS ITEMS *** However, you will have plenty of chances to review the work before commiting any changes.

**Where this notebook can run**  
- ArcGIS Online Notebook (JupyterLab-style).
- VS Code on macOS with a local Jupyter kernel.
- VS Code on Windows with a local Jupyter kernel.

**How to use this notebook**  
 - Click on the text "Setup and authenticate" below. 
 - There are two types of cells, Markdown (formatted notes) and Code.
 - An indicator -- typically a vertical blue line -- should highlight that you have selected the "Setup and authenticate" Markdown cell.
 - Once selected, click the "Play" button in the toolbar above to run the cell and advance to the next Code cell.
 - Click the "Play" button a second time to run the code cell.
 - After several seconds a "Setup Notebook" button should appear. Click the button to begin setup and authentication.
 - After each cell completes, click the text within the following Markdown cell.
 - Click the "Play" button to advance to the Code cell, then click the "Play" button a second time to make a button appear.
 - Click the button to run the code in the cell. 

**Notes**  
- Organization-wide scans can take time, especially in large orgs, so progress messages are shown as users are processed.
- You can monitor the status of long running cells by viewing the small circle in the top right of the page.
- If you click on a code cell it will expand showing you the behind-the-scenes Python code.
- For a cleaner interface select View > Collapse All Code in the menu bar above to hide the code .
- If at any point you get stuck and want to start over, just click Kernel > Restart Kernel and Clear Outputs of All Cells... in the menu bar
- The workflow is designed to be safe by default: review first, then update.


**TL;DR**


In [ ]:
# Run this cell to display Notebook details
from IPython.display import display, Markdown

# Display details of what this notebook does
tldr_md = """
**What this notebook does**  
- Authenticates to ArcGIS Online
- Scans an entire Organization's Item Description page's "Terms of Use" section (aka `licenseInfo`).
- Finds items that match one or more target strings (for example, outdated logo URLs or legacy text snippets).
- Exports scan outputs for auditability (default filenames: `scan_matches.csv` and `scan_errors.csv`).
- Supports optional secondary scans to target additional terms while excluding already matched item IDs. (improves scan speed and accuracy)
- Builds a dry-run update plan that shows old vs new HTML before any edit is applied.
- Generates a user-friendly side-by-side HTML review report for visual QA.
- Applies updates only after explicit confirmation, then exports success and error results.
- Works in local VS Code notebooks (macOS and Windows) and ArcGIS Online notebooks.
"""
display(Markdown(tldr_md))

## 1. Setup and authenticate
Write the bundled helper files into the runtime, then initialize the notebook environment and connect to ArcGIS Online.


In [1]:
# Bootstrap bundled assets for the portable notebook.import base64import sysfrom pathlib import PathOUTPUT_DIR_NAME = "notebook_outputs"RUNTIME_ROOT = Path("/arcgis/home") if Path("/arcgis/home").exists() else Path.cwd()RUNTIME_DIR = (RUNTIME_ROOT / OUTPUT_DIR_NAME).resolve()RUNTIME_DIR.mkdir(parents=True, exist_ok=True)HELPER_FUNCTIONS_B64 = (    "..."    "dHVybiBzdWNjZXNzX2RmLCBlcnJvcnNfZGY=")ESRI_TOU_HTML_B64 = (    "PHA+CiAgICA8aW1nIHNyYz0iaHR0cHM6Ly93d3cuZXNyaS5jb20vY29udGVudC9kYW0vZXNyaXNpdGVzL2VuLXVzL2NvbW1vbi9sb2dvcy9lc3JpLWxvZ28u"    "anBnIiBhbHQ9IkVzcmkgbG9nbyIgd2lkdGg9IjExMyIgaGVpZ2h0PSIzOSI+CjwvcD4KPHAgc3R5bGU9ImNvbG9yOnJnYig3NCw3NCw3NCk7Zm9udC1mYW1p"    "bHk6J0F2ZW5pciBOZXh0JyxBdmVuaXIsJ0hlbHZldGljYSBOZXVlJyxzYW5zLXNlcmlmO2ZvbnQtc2l6ZToxNnB4O21hcmdpbjowIDAgMXJlbTsiPgogICAg"    "VGhpcyB3b3JrIGlzIGxpY2Vuc2VkIHVuZGVyIHRoZSBFc3JpIE1hc3RlciBMaWNlbnNlIEFncmVlbWVudC4KPC9wPgo8cCBzdHlsZT0iY29sb3I6cmdiKDc0"    "LDc0LDc0KTtmb250LWZhbWlseTonQXZlbmlyIE5leHQnLEF2ZW5pciwnSGVsdmV0aWNhIE5ldWUnLHNhbnMtc2VyaWY7Zm9udC1zaXplOjE2cHg7bWFyZ2lu"    "OjA7Ij4KICAgIDxhIHN0eWxlPSJjb2xvcjpyZ2IoMCw5NywxNTUpO3RleHQtZGVjb3JhdGlvbjpub25lOyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVu"    "ZXIgbm9yZWZlcnJlciIgaHJlZj0iaHR0cHM6Ly9nb3RvLmFyY2dpcy5jb20vdGVybXNvZnVzZS92aWV3c3VtbWFyeSI+PHN0cm9uZz5WaWV3IFN1bW1hcnk8"    "L3N0cm9uZz48L2E+IHwgPGEgc3R5bGU9ImNvbG9yOnJnYigwLDk3LDE1NSk7dGV4dC1kZWNvcmF0aW9uOm5vbmU7IiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJu"    "b29wZW5lciBub3JlZmVycmVyIiBocmVmPSJodHRwczovL2dvdG8uYXJjZ2lzLmNvbS90ZXJtc29mdXNlL3ZpZXd0ZXJtc29mdXNlIj48c3Ryb25nPlZpZXcg"    "VGVybXMgb2YgVXNlPC9zdHJvbmc+PC9hPgo8L3A+")BOOTSTRAP_FILES = {    "helper_functions.py": base64.b64decode(HELPER_FUNCTIONS_B64).decode("utf-8"),    "Esri_ToU.html": base64.b64decode(ESRI_TOU_HTML_B64).decode("utf-8"),}for filename, file_text in BOOTSTRAP_FILES.items():    target_path = RUNTIME_DIR / filename    target_path.write_text(file_text, encoding="utf-8")    print(f"Bootstrapped asset: {target_path}")if str(RUNTIME_DIR) not in sys.path:    sys.path.insert(0, str(RUNTIME_DIR))print(f"Portable notebook assets are ready in: {RUNTIME_DIR}")# When running in ArcGIS Online, you can select View > Collapse All Code in the toolbar above to hide the code for a cleaner view.
print("Initializing...")

# Cell 1. Import packages, configure paths, authenticate, and load helper functions
import sys
from pathlib import Path
import pandas as pd
import ipywidgets as widgets

# Support local VS Code and ArcGIS Online paths; prefer bootstrapped notebook_outputs first.
NOTEBOOK_DIR = Path.cwd()
CANDIDATE_HELPER_DIRS = [
    NOTEBOOK_DIR / "notebook_outputs",
    NOTEBOOK_DIR,
    Path("/arcgis/home/notebook_outputs"),
    Path("/arcgis/home"),
]
for p in CANDIDATE_HELPER_DIRS:
    helper_file = p / "helper_functions.py"
    if helper_file.exists() and str(p) not in sys.path:
        sys.path.append(str(p))

from helper_functions import (
    detect_environment,
    default_output_dir_str,
    default_output_path_str,
    initialize_ui,
    set_runtime_context,
    bind_button_with_status,
    setup_notebook_btn,
    run_primary_scan_btn,
    save_scan_outputs_btn,
    load_previous_scan_btn,
    run_secondary_scan_btn,
    run_strict_match_filter_btn,
    run_dry_run_with_file_btn,
    create_report_btn,
    export_dry_run_btn,
    apply_updates_btn,
    export_final_results_btn,
    OFFICIAL_TOU_HTML_FILE,
 )

# Set Pandas dataframe display options
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", 1000)

# Define shared notebook state
context = {
    "gis": None,
    "matches_df": None,
    "errors_df": None,
    "all_items_df": None,
    "TARGET_STRINGS": [],
    "output_dir": default_output_dir_str(),
    "official_tou_html_file": OFFICIAL_TOU_HTML_FILE,
}
set_runtime_context(context)

# Detect the current environment
current_env, env_string = detect_environment()
print(f"Currently running in {env_string}")
print(f"Default output folder: {context['output_dir']}")

output1 = initialize_ui("output")
context["output1"] = output1

# Create widgets
btn1 = initialize_ui("button", description="Setup Notebook", width="200px")
status1 = widgets.HTML(value="")
context["status1"] = status1
display(widgets.HBox([btn1, status1]))
bind_button_with_status(
    btn1,
    setup_notebook_btn,
    "status1",
    "Setup in progress... please wait.",
    "Setup complete.",
    "Setup failed. See details below.",
    output_key="output1",
)
display(output1)

Initializing...
Currently running in VSCode Notebook environment
Default output folder: /Users/davi6569/Documents/GitHub/AGO-item-description-editor/notebook_outputs


Button(description='Setup Notebook', layout=Layout(height='40px', width='200px'), style=ButtonStyle())

Output()

## 2. Scan your content
Search your content for the text strings and/or HTML entered below.


In [2]:
# Cell 2: Define terms and scan org content for licenseInfo matches
output2 = initialize_ui("output")
context["output2"] = output2

help2 = widgets.HTML(
    value=(
        "<div style='margin:0; padding:0; line-height:1.25;'>"
        "<strong>Enter text or HTML to find in the Terms of Use section.</strong> "
        "Use CSV-style input (comma-separated).<br>"
        "Wrap text with spaces in double quotes, for example: "
        "&quot;&lt;a href=https://example.com&gt;link&lt;/a&gt;&quot;.<br>"
        "Formatting will automatically be bundled for proceesing when you click the button."
        "</div>"
    )
)

input2 = initialize_ui(
    "textarea",
    value='https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png, esrilogo',
    description="",
    width="700px",
    height="70px",
)
context["input2"] = input2
btn2 = initialize_ui("button", description="Scan for items", width="200px")
status2 = widgets.HTML(value="")
context["status2"] = status2

display(widgets.VBox([help2, input2, widgets.HBox([btn2, status2]), output2]))

bind_button_with_status(
    btn2,
    run_primary_scan_btn,
    "status2",
    "Scan in progress... please wait.",
    "Scan complete.",
    "Scan failed. See details below.",
    output_key="output2",
)

## 3. Save scan results
Save the latest scan output to CSV files. You can rename the files or change where they are written.


In [ ]:
# Cell 3: Save latest scan outputs to CSV
output3 = initialize_ui("output")
context["output3"] = output3
input3_matches = initialize_ui(
    "text",
    value=default_output_path_str("scan_matches.csv"),
    description="Matches CSV:",
    width="700px",
)
context["input3_matches"] = input3_matches
input3_errors = initialize_ui(
    "text",
    value=default_output_path_str("scan_errors.csv"),
    description="Errors CSV:",
    width="700px",
)
context["input3_errors"] = input3_errors
input3_all_items = initialize_ui(
    "text",
    value=default_output_path_str("scan_all_items.csv"),
    description="All items CSV:",
    width="700px",
)
context["input3_all_items"] = input3_all_items
btn3 = initialize_ui("button", description="Save files")
status3 = widgets.HTML(value="")
context["status3"] = status3
display(widgets.VBox([input3_matches, input3_errors, input3_all_items, widgets.HBox([btn3, status3]), output3]))

bind_button_with_status(
    btn3,
    save_scan_outputs_btn,
    "status3",
    "Save in progress... please wait.",
    "Save complete.",
    "Save failed. See details below.",
    output_key="output3",
)

## 4. Optionally reload results from a previous scan
Load previously saved CSV files so you can continue working without running a new scan.


In [ ]:
# Cell 4: Optionally load results from a previous scan
output4 = initialize_ui("output")
context["output4"] = output4

input4_matches = initialize_ui(
    "text",
    value=default_output_path_str("scan_matches.csv"),
    description="Matches CSV:",
    width="900px",
)
context["input4_matches"] = input4_matches
input4_errors = initialize_ui(
    "text",
    value=default_output_path_str("scan_errors.csv"),
    description="Errors CSV:",
    width="900px",
)
context["input4_errors"] = input4_errors
input4_all_items = initialize_ui(
    "text",
    value=default_output_path_str("scan_all_items.csv"),
    description="All items CSV:",
    width="900px",
)
context["input4_all_items"] = input4_all_items
btn4 = initialize_ui("button", description="Load previous scan files", width="240px")
status4 = widgets.HTML(value="")
context["status4"] = status4

display(
    widgets.VBox([
        input4_matches,
        input4_errors,
        input4_all_items,
        widgets.HBox([btn4, status4]),
        output4,
    ])
)

bind_button_with_status(
    btn4,
    load_previous_scan_btn,
    "status4",
    "Load in progress... please wait.",
    "Load complete.",
    "Load failed. See details below.",
    output_key="output4",
)

## 5. Secondary scan
This cell saves you time if you want to search additional terms.
If you want to search again, click the SECONDARY_SCAN check box and add the new terms to NEW_TERMS and run the cell below


In [ ]:
# Cell 5: Optional secondary scan using new terms and excluding previous matches
output5 = initialize_ui("output")
context["output5"] = output5
checkbox5 = initialize_ui("checkbox", description="Run secondary scan with new search terms?", value=False)
context["checkbox5"] = checkbox5
help5 = widgets.HTML(
    value=(
        "<div style='margin:0; padding:0; line-height:1.25;'>"
        "As above, use CSV-style input separated by commas.<br>"
        "Wrap text with spaces in double quotes, for example: &quot;text with spaces&quot;."
        "</div>"
    )
)
input5 = initialize_ui(
    "textarea",
    value='https://www.esri.com/content/dam/esrisites/en-us/common/logos/esri-logo.jpg',
    description="",
    width="700px",
    height="70px",
)
context["input5"] = input5

btn5 = initialize_ui("button", description="Run secondary scan")
status5 = widgets.HTML(value="")
context["status5"] = status5
display(widgets.VBox([checkbox5, help5, input5, widgets.HBox([btn5, status5]), output5]))

bind_button_with_status(
    btn5,
    run_secondary_scan_btn,
    "status5",
    "Secondary scan in progress... please wait.",
    "Secondary scan complete.",
    "Secondary scan failed. See details below.",
    output_key="output5",
)

## 6. Optionally narrow your original query


In [ ]:
# Cell 6: Optionally filter the scan result to rows containing the exact text below
output6 = initialize_ui("output")
context["output6"] = output6
input6 = initialize_ui(
    "text",
    value="https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png",
    description="Exact text:",
    width="700px",
)
context["input6"] = input6
btn6 = initialize_ui("button", description="Filter exact matches", width="200px")
status6 = widgets.HTML(value="")
context["status6"] = status6
display(widgets.VBox([input6, widgets.HBox([btn6, status6]), output6]))

bind_button_with_status(
    btn6,
    run_strict_match_filter_btn,
    "status6",
    "Filter in progress... please wait.",
    "Filter complete.",
    "Filter failed. See details below.",
    output_key="output6",
)

## 7. Dry run


In [ ]:
# Cell 7: Do a dry-run before making any changes. Modify the input file to use your own custom HTML.
output7 = initialize_ui("output")
context["output7"] = output7
input7 = initialize_ui(
    "text",
    value=context.get("official_tou_html_file", OFFICIAL_TOU_HTML_FILE),
    description="Input HTML file:",
    width="900px",
)
context["input7"] = input7
btn7 = initialize_ui("button", description="Build dry run", width="180px")
status7 = widgets.HTML(value="")
context["status7"] = status7

display(widgets.VBox([input7, widgets.HBox([btn7, status7]), output7]))

bind_button_with_status(
    btn7,
    run_dry_run_with_file_btn,
    "status7",
    "Dry run in progress... please wait.",
    "Dry run complete.",
    "Dry run failed. See details below.",
    output_key="output7",
)

## 8. Export dry run results


In [ ]:
# Cell 8: Export the dry-run plan for record-keeping and review
output8 = initialize_ui("output")
context["output8"] = output8
input8_csv_name = initialize_ui(
    "text",
    value=default_output_path_str("dry_run_results.csv"),
    description="Output CSV:",
    width="700px",
)
context["input8_csv_name"] = input8_csv_name
btn8 = initialize_ui("button", description="Export dry-run CSV", width="200px")
status8 = widgets.HTML(value="")
context["status8"] = status8
display(widgets.VBox([input8_csv_name, widgets.HBox([btn8, status8]), output8]))

bind_button_with_status(
    btn8,
    export_dry_run_btn,
    "status8",
    "Dry-run export in progress... please wait.",
    "Dry-run export complete.",
    "Dry-run export failed. See details below.",
    output_key="output8",
)

## 9. Create report


In [ ]:
# Cell 9: Generate an HTML report for review before updating items
output9 = initialize_ui("output")
context["output9"] = output9
input9_report_name = initialize_ui(
    "text",
    value=default_output_path_str("dry_run_report.html"),
    description="Report HTML:",
    width="700px",
)
context["input9_report_name"] = input9_report_name
input9_selection_json = initialize_ui(
    "text",
    value="selected_item_ids.json",
    description="Filename generated when downloading IDs after review:",
    width="700px",
)
context["input9_selection_json"] = input9_selection_json
btn9 = initialize_ui("button", description="Create report", width="200px")
status9 = widgets.HTML(value="")
context["status9"] = status9

display(
    widgets.VBox([
        input9_report_name,
        input9_selection_json,
        widgets.HBox([btn9, status9]),
        output9,
    ])
)

bind_button_with_status(
    btn9,
    create_report_btn,
    "status9",
    "Report generation in progress... please wait.",
    "Report generation complete.",
    "Report generation failed. See details below.",
    output_key="output9",
)

## 10. Commit updates


In [ ]:
# Cell 10: Apply updates only after reviewing the dry run report 
output10 = initialize_ui("output")
context["output10"] = output10
input10_ids = initialize_ui(
    "text",
    value="selected_item_ids.json",
    description="JSON file with item IDs to update:",
    width="700px",
)
context["input10_ids"] = input10_ids

input10_confirm = initialize_ui(
    "text",
    value="",
    description="Type APPLY UPDATES to confirm:",
    width="700px",
)
context["input10_confirm"] = input10_confirm

btn10 = initialize_ui("button", description="Execute update", width="180px")
status10 = widgets.HTML(value="")
context["status10"] = status10
display(widgets.VBox([input10_ids, input10_confirm, widgets.HBox([btn10, status10]), output10]))

bind_button_with_status(
    btn10,
    apply_updates_btn,
    "status10",
    "Update in progress... please wait.",
    "Update complete.",
    "Update failed. See details below.",
    output_key="output10",
)

## 11. Export final results


In [ ]:
# Cell 11: Export final update results to CSV files for record-keeping
output11 = initialize_ui("output")
context["output11"] = output11
input11_success_csv = initialize_ui(
    "text",
    value=default_output_path_str("update_successes.csv"),
    description="Success CSV:",
    width="700px",
)
context["input11_success_csv"] = input11_success_csv
input11_errors_csv = initialize_ui(
    "text",
    value=default_output_path_str("update_errors.csv"),
    description="Errors CSV:",
    width="700px",
)
context["input11_errors_csv"] = input11_errors_csv
btn11 = initialize_ui("button", description="Export final CSVs", width="180px")
status11 = widgets.HTML(value="")
context["status11"] = status11
display(widgets.VBox([input11_success_csv, input11_errors_csv, widgets.HBox([btn11, status11]), output11]))

bind_button_with_status(
    btn11,
    export_final_results_btn,
    "status11",
    "Final export in progress... please wait.",
    "Final export complete.",
    "Final export failed. See details below.",
    output_key="output11",
)